In [ ]:
# run this notebook after 2_generate_hgdp_1kg_pcs_python 
# use this notebook to project AoU samples onto HGDP+1KG PC-space
# after this, run 4_make_loadings_ukb_style_R

In [ ]:
import os
import pandas as pd
bucket=os.getenv("WORKSPACE_BUCKET")
import hail as hl

In [ ]:
mt = hl.read_matrix_table('gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt')

In [ ]:
pca_variants_final_ht = hl.import_table(f'{bucket}/pca/pca_variants_acaf.tsv')
pca_variants_final_ht = pca_variants_final_ht.annotate(
    locus = hl.locus(pca_variants_final_ht.chr, hl.int(pca_variants_final_ht.pos), reference_genome='GRCh38'),
    alleles = hl.parse_json(pca_variants_final_ht.final_alleles, hl.tarray(hl.tstr)))
pca_variants_final_ht = pca_variants_final_ht.key_by('locus', 'alleles')

In [ ]:
mt = mt.filter_rows(hl.is_defined(pca_variants_final_ht[mt.row_key]))
mt = mt.naive_coalesce(5000)
out_path = f'{bucket}/pca/aou_pca_biallelic.mt'
mt.write(out_path, overwrite = True)

In [ ]:
mt_hgdp_1000g = hl.read_matrix_table(f'{bucket}/pca/subset_hgdp_1kg_unrelated_outliers_removed_acaf.mt')
mt_hgdp_1000g = mt_hgdp_1000g.annotate_rows(af=hl.agg.mean(mt_hgdp_1000g.GT.n_alt_alleles()) / 2)
mt_hgdp_1000g.write(f'{bucket}/pca/subset_hgdp_1kg_unrelated_outliers_removed_acaf_af.mt', overwrite = True)

In [ ]:
loadings_ht = hl.import_table(f'{bucket}/pca/hgdp_1000g_variant_loadings_acaf.tsv', impute=True)
split_vid = loadings_ht.locus.split(":")
loadings_ht = loadings_ht.annotate(
    locus_new = hl.locus(split_vid[0], hl.int(split_vid[1]), reference_genome='GRCh38'),
    alleles_new = hl.parse_json(loadings_ht.alleles, hl.tarray(hl.tstr)),
    loadings = hl.parse_json(loadings_ht.loadings, hl.tarray(hl.tfloat64)))
loadings_ht = loadings_ht.key_by('locus_new', 'alleles_new')                           
loadings_ht = loadings_ht.annotate(af=mt_hgdp_1000g.rows()[loadings_ht.key].af)   
loadings_ht.export(f'{bucket}/pca/hgdp_1000g_variant_loadings_acaf_af_added.tsv') 

In [ ]:
loadings_ht = hl.import_table(f'{bucket}/pca/hgdp_1000g_variant_loadings_acaf_af_added.tsv', impute=True)
split_vid = loadings_ht.locus.split(":")
loadings_ht = loadings_ht.annotate(
    locus_new = hl.locus(split_vid[0], hl.int(split_vid[1]), reference_genome='GRCh38'),
    alleles_new = hl.parse_json(loadings_ht.alleles, hl.tarray(hl.tstr)),
    loadings = hl.parse_json(loadings_ht.loadings, hl.tarray(hl.tfloat64)))
loadings_ht = loadings_ht.key_by('locus_new', 'alleles_new') 

In [ ]:
mt = hl.read_matrix_table(f'{bucket}/pca/aou_pca_biallelic.mt')
sample_ids = mt.s.collect()
chunks = [sample_ids[i::10] for i in range(10)]
# Save each chunk to a separate directory
for i in range(10):
    print(i)
    chunk = chunks[i]
    mt_part = mt.filter_cols(hl.literal(chunk).contains(mt.s))
    mt_part.write(f'{bucket}/pca/aou_pca_annotated_biallelic_{i}.mt', overwrite=True)   
    ht = hl.experimental.pc_project(mt_part.GT, loadings_ht.loadings, loadings_ht.af)
    ht.export(f'{bucket}/pca/aou_pcs_biallelic_{i}.tsv') 

In [ ]:
%%bash 

gsutil cp $WORKSPACE_BUCKET/pca/hgdp_1000g_variant_loadings_acaf_af_added.tsv ./pca/
gsutil cp $WORKSPACE_BUCKET/pca/pca_variants_acaf.tsv ./pca/
gsutil cp $WORKSPACE_BUCKET/pca/aou_pcs_biallelic* ./pca/ 
wget https://storage.googleapis.com/gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg_v2/metadata_and_qc/gnomad_meta_updated.tsv .